[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/halla-ai/deepnlp-2026/blob/main/notebooks/week-13.ipynb)

# 13주차: 논문 방법의 작은 CPU 재현

- 관찰 전용: 모든 셀을 수정 없이 실행, 학생 알고리즘 구현 없음
- Self-Refine §2의 생성→피드백→수정 한 회 구조를 제공 코드로 실행
- 실제 공개 모델 생성, 교수 작성 점수나 모의 모델 호출 없음
- Qwen3-0.6B-Base CPU float32, 두 연산 스레드, greedy, 호출당 최대 24 생성 토큰
- 모델 가중치 다운로드와 인터넷 필요. 모델 로딩·설치는 비교 지연에서 제외하고 따로 기록
- 원 논문의 모델·과제·few-shot 프롬프트·반복 중단 조건과 다름. 원 성능 수치·학습·SAIGE 재현 아님
- Base 모델은 지시 학습 모델과 다름. 형식 실패도 결과로 유지
- [Self-Refine (2023), §2·§4·§6](https://arxiv.org/html/2303.17651v2)
- [Rethinking Multi-Agent Collaboration (2026), §3·§5](https://arxiv.org/html/2609.19759v1): 읽기용, 해당 시스템 구현 없음
- [Qwen3 Technical Report (2025)](https://arxiv.org/abs/2505.09388), [모델 카드](https://huggingface.co/Qwen/Qwen3-0.6B-Base)


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "torch>=2.6,<3", "transformers>=4.51,<6"])


## 1. 고정판의 공개 근거 수집

- 공개 행정시 사실을 실제 문단과 대조. 원문 전체 파일 저장 없음
- 고정판과 제목 불일치·네트워크 오류는 명시적으로 실패, 가짜 fallback 없음
- CC BY-SA 4.0. 문서 기여자와 이용 조건은 원문·이력·이용약관 참조


In [ ]:
import hashlib
import html
import json
import math
import re
import time
import urllib.error
import urllib.parse
import urllib.request
from datetime import datetime, timezone
from pathlib import Path

OLDID = 42464087
SOURCE = f"https://ko.wikipedia.org/w/index.php?oldid={OLDID}"
api = "https://ko.wikipedia.org/w/api.php?" + urllib.parse.urlencode({
    "action": "parse", "oldid": OLDID, "prop": "text", "format": "json", "formatversion": 2})
request = urllib.request.Request(api, headers={"User-Agent": "deepnlp-course/1.0 (yj.lee@chu.ac.kr)"})
for attempt in range(3):
    try:
        with urllib.request.urlopen(request, timeout=40) as response:
            raw_response = response.read()
        parsed = json.loads(raw_response)["parse"]
        break
    except (urllib.error.URLError, TimeoutError) as error:
        if attempt == 2:
            raise RuntimeError("공개 원문 수집 실패. 연결·출처 확인 후 다시 실행. 대체 문서 없음") from error
        time.sleep(2 ** attempt)
assert parsed["revid"] == OLDID, "고정판 번호 불일치"
assert parsed["title"] == "제주특별자치도", "원문 제목 불일치"

def clean(fragment):
    fragment = re.sub(r"<sup\b[^>]*>.*?</sup>", "", fragment, flags=re.S)
    fragment = re.sub(r"<[^>]+>", " ", fragment)
    return re.sub(r"\s+", " ", html.unescape(fragment)).strip()

paragraphs = [clean(p) for p in re.findall(r"<p\b[^>]*>(.*?)</p>", parsed["text"], flags=re.S)]
evidence = [p for p in paragraphs if all(term in p for term in ("행정시", "제주시", "서귀포시"))]
assert evidence, "두 행정시의 원문 문단을 확인하지 못함. 자동 대체 금지"
source_metadata = {
    "url": SOURCE, "title": parsed["title"], "revision": parsed["revid"],
    "fetched_at_utc": datetime.now(timezone.utc).isoformat(),
    "response_sha256": hashlib.sha256(raw_response).hexdigest(),
    "attribution": "한국어 위키백과 제주특별자치도 문서와 해당 판의 기여자; 저자·이용 조건은 문서 이력 참조",
}
print("출처 메타데이터:", source_metadata)
print("실제 공개 원문 근거:", evidence[0])
source_metadata['license'] = 'CC BY-SA 4.0'
source_metadata['history_url'] = 'https://ko.wikipedia.org/w/index.php?title=제주특별자치도&action=history'
source_metadata['terms_url'] = 'https://foundation.wikimedia.org/wiki/Policy:Terms_of_Use'
context = evidence[0]
print('근거 문자 수:', len(context))


## 2. 모델과 실행 조건 고정

- 가중치 revision 고정. CPU, float32, eval, gradient 없음
- 실제 파라미터 수·가중치 바이트·라이브러리·CPU·모델 로드 시간을 출력
- 가중치 바이트는 RSS가 아님. KV 캐시·토크나이저·중간 텐서·운영체제 메모리 별도


In [ ]:
import platform
import torch, transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL = 'Qwen/Qwen3-0.6B-Base'
REVISION = 'da87bfb608c14b7cf20ba1ce41287e8de496c0cd'
torch.set_num_threads(2)
torch.manual_seed(0)
t0 = time.perf_counter()
tok = AutoTokenizer.from_pretrained(MODEL, revision=REVISION)
model = AutoModelForCausalLM.from_pretrained(MODEL, revision=REVISION, torch_dtype=torch.float32).to('cpu').eval()
load_sec = time.perf_counter() - t0
parameter_count = sum(p.numel() for p in model.parameters())
weight_bytes = sum(p.numel()*p.element_size() for p in model.parameters())
conditions = dict(model=MODEL, revision=REVISION, device='cpu', dtype='float32', threads=2,
                 max_new_tokens=24, do_sample=False, seed=0, torch=torch.__version__,
                 transformers=transformers.__version__, python=platform.python_version(),
                 machine=platform.machine(), load_sec=load_sec,
                 parameter_count=parameter_count, weight_bytes=weight_bytes)
print(json.dumps(conditions, ensure_ascii=False, indent=2))


## 3. 제공된 생성·검토·수정 함수

- 프롬프트 세 종류가 같은 모델 가중치 공유. 서로 다른 독립 모델·학습 에이전트가 아님
- 예열 한 호출의 시간과 토큰은 비교 합계에서 제외
- 호출 시간: 토큰화 이후 generate 구간 wall time. 프리필+디코딩 포함, 순수 디코딩 시간 아님
- 로그에는 프롬프트 hash·출력·토큰·시간만 저장. 공개 원문·전체 프롬프트 저장 없음


In [ ]:
def ask(prompt, stage):
    batch = tok(prompt, return_tensors='pt')
    started = time.perf_counter()
    with torch.inference_mode():
        out = model.generate(**batch, max_new_tokens=24, do_sample=False,
                             pad_token_id=tok.eos_token_id, use_cache=True)
    elapsed = time.perf_counter() - started
    new_ids = out[0, batch.input_ids.shape[1]:]
    return dict(stage=stage, text=tok.decode(new_ids, skip_special_tokens=True).strip(),
                input_tokens=int(batch.input_ids.shape[1]), output_tokens=int(new_ids.numel()),
                generate_sec=elapsed, prompt_sha256=hashlib.sha256(prompt.encode()).hexdigest())

warmup = ask('제주특별자치도의 행정시는', 'warmup')
print('예열(비교 제외):', warmup)
cases = [
    dict(id='cities', question='제주특별자치도의 두 행정시 이름은?', expected=['제주시','서귀포시']),
    dict(id='count', question='제주특별자치도의 행정시 수는?', expected=['2']),
    dict(id='other', question='제주특별자치도에서 제주시 외의 행정시 이름은?', expected=['서귀포시']),
]
def base_prompt(case):
    return f"문서: {context}\n질문: {case['question']}\n문서에 있는 정답만 한 줄로 쓰시오. 수는 아라비아 숫자로 쓰시오.\n답:"

def match(text, case):
    # 첫 줄의 답 형식만 평가. 한국어 별칭·설명·부정문은 자동으로 정답 인정하지 않음.
    first = text.splitlines()[0].strip() if text.strip() else ''
    normalized = re.sub(r'[\s,·/;:.]', '', first)
    if case['id'] == 'cities':
        return normalized in ('제주시서귀포시', '서귀포시제주시')
    return normalized == case['expected'][0]

assert match('제주시, 서귀포시', cases[0])
assert not match('제주시와 서귀포시가 아닙니다', cases[0])
assert not match('서귀포시는 아닙니다', cases[2])


## 4. 동일 초안에서 한 번의 Self-Refine 실행

- 먼저 direct 초안 생성, 초안과 근거로 피드백 생성, 근거·초안·피드백으로 수정 생성
- direct와 refine 구성은 같은 초안을 공유. refine 비교 비용에는 초안 호출을 포함
- 총 비교용 실행은 9호출(3질문×3단계). 두 구성을 각각 운영하면 질문당 1호출 대 3호출
- 문맥과 이전 출력이 늘어난 효과, 프롬프트 형식, 모델 능력이 함께 바뀌므로 독립 에이전트의 인과 효과 판정 불가


In [ ]:
runs = []
for case in cases:
    initial = ask(base_prompt(case), 'initial')
    feedback = ask(f"문서: {context}\n질문: {case['question']}\n초안: {initial['text']}\n문서와 초안을 대조하고 수정할 점 한 가지만 짧게 쓰시오.\n검토:", 'feedback')
    refined = ask(f"문서: {context}\n질문: {case['question']}\n초안: {initial['text']}\n검토: {feedback['text']}\n검토를 참고해 정답만 한 줄로 다시 쓰시오. 수는 아라비아 숫자로 쓰시오.\n답:", 'refined')
    rows = [initial, feedback, refined]
    direct = dict(calls=1, input_tokens=initial['input_tokens'], output_tokens=initial['output_tokens'],
                  generate_sec=initial['generate_sec'], format_match=match(initial['text'],case))
    refine = dict(calls=3, input_tokens=sum(r['input_tokens'] for r in rows),
                  output_tokens=sum(r['output_tokens'] for r in rows),
                  generate_sec=sum(r['generate_sec'] for r in rows), format_match=match(refined['text'],case))
    row = dict(case_id=case['id'], question=case['question'], expected=case['expected'],
               stages=rows, direct=direct, refine=refine)
    runs.append(row)
    print(json.dumps(row, ensure_ascii=False, indent=2))
assert len(runs) == 3 and all(len(r['stages'])==3 for r in runs)


## 5. 구성 비교표와 오답 전이

- 점수는 지정 첫 줄 형식 일치. 의미적 사실 정확도·faithfulness·벤치마크 성능과 다름
- 정답을 놓친 형식 오류는 출력과 함께 수동 검토. 규칙을 결과 확인 후 유리하게 바꾸지 않음
- 비용은 토큰·호출·시간의 대리 지표. 현금 비용·전력·RSS·서버 동시 처리량은 미측정
- 3개 질문은 통계적 우위 주장에 부족. 같은 조건 repeated timing·다른 문서·더 많은 질문은 후속 연구


In [ ]:
comparison = []
for method in ('direct','refine'):
    metrics = [r[method] for r in runs]
    row = dict(method=method, calls=sum(x['calls'] for x in metrics),
               input_tokens=sum(x['input_tokens'] for x in metrics),
               output_tokens=sum(x['output_tokens'] for x in metrics),
               generate_sec=sum(x['generate_sec'] for x in metrics),
               format_matches=sum(x['format_match'] for x in metrics), n=len(metrics),
               cash_cost=None, peak_rss=None, energy=None)
    comparison.append(row)
    print('구성 비교:', row)
transitions = {key:0 for key in ('유지 정답','교정','퇴행','유지 실패')}
for r in runs:
    before, after = r['direct']['format_match'], r['refine']['format_match']
    key = '유지 정답' if before and after else '교정' if after else '퇴행' if before else '유지 실패'
    transitions[key] += 1
    print(r['case_id'], key, '초안:',r['stages'][0]['text'], '수정:',r['stages'][2]['text'])
assert sum(transitions.values()) == 3
print('전이:',transitions)


## 6. 결과 저장, 한 장 기록, 확인 질문

| 기록 항목 | 채울 근거 |
|---|---|
| 주장 | 동일 모델 한 회 피드백·수정이 이 세 질문에서 어떤 변화인지 |
| 방법 | §2 생성·피드백·수정과 코드 4절의 대응 |
| 조건 | revision, CPU, dtype, threads, 최대 생성, 고정판 |
| 근거 | 각 초안·피드백·수정, 형식 점수 전이, 비교 표 |
| 차이 | 논문 모델·few-shot·과제·중단 조건과의 차이 |
| 한계 | 작은 표본, Base 모델, 형식 채점, 순서·시간 변동 |
| 적용 판단 | GPU 없는 조직의 메모리·응답 대기·사람 검토 조건 |

1. 같은 초안에서 피드백을 붙였을 때 답이 좋아졌는지 출력으로 확인
2. 수정 후 실패하면 피드백 오류·수정 반영 실패·출력 형식 오류를 구분
3. 호출 수 비율과 시간 비율이 다른 이유를 토큰 표로 설명
4. 동일 가중치 역할 분리가 독립 검토자의 근거가 되는지 설명

- 산출물: JSON과 위 한 장 기록, 별도 제출 공지가 있을 때 사용
- 이름·연락처 없이 학번 폴더만 사용, 원문 전체·API 키 제출 금지
- 실제 모델 출력은 공개 원문의 일부 문장을 인용할 수 있음. 원문 CC BY-SA 출처·기여자·이용조건 메타데이터를 결과와 함께 보존


In [ ]:
report = dict(kind='Self-Refine one-cycle structural reproduction, not paper benchmark reproduction',
              source_metadata=source_metadata, conditions=conditions,
              runs=runs, comparison=comparison, transitions=transitions,
              limitations=['3 factual questions','base model','strict first-line format match',
                           'single timing run','no independent models','no SAIGE implementation'])
path = Path('week-13-observations.json')
path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
assert 'context' not in report and 'evidence' not in report
print('저장:', path)
